# Hospital Operations Analysis

---
## Sources and links 

Kaggle Dataset : [Hospital Management Dataset](https://www.kaggle.com/datasets/kanakbaghel/hospital-management-dataset?select=appointments.csv)

Website used to create ERD : 

---
## Overview 
- title 
- sources and links
- overview
- imports
    - library imports
    - data imports
- business understanding 
    - business objectives 
    - situation assesment 
    - project plan
- data understanding
    - data properties 
    - ERD
- methodoligy
- functions
- database manegement
    - clear old databse 
    - create databse connection
    - create tables
    - data insertion and validation

---
## Imports

### Library Imports

In [23]:
import pandas as pd
import sqlite3
import matplotlib as plt
from pathlib import Path
from datetime import datetime

In [24]:
# this cell is to show the last update to this notebook
now = datetime.now()
print(f'This notebook was last updated at : {now} PST')

This notebook was last updated at : 2026-07-14 11:39:35.694463 PST


### Data Imports 

In [2]:
patients_df = pd.read_csv('operations_data/patients.csv')
doctors_df = pd.read_csv('operations_data/doctors.csv')
appointments_df = pd.read_csv('operations_data/appointments.csv')
billing_df = pd.read_csv('operations_data/billing.csv')
treatment_csv = pd.read_csv('operations_data/treatments.csv')

---
## Database Manegement 
Here we are firstly going to clear this notebooks databse, as this prevents data being du

In [3]:
# delete old database
db_path = Path('hospital.db')

if db_path.exists():
    db_path.unlink()

Then create an empty databse for us to add tables and eventually populate those tables with the data from the `.csv` file. This is done this way so we can have much more control over the database. 

In [4]:
conn = sqlite3.connect('hospital.db')
conn.execute('PRAGMA foreign_keys = ON')

### Tables
Here we create empty tables with the constraints we want for each data point. We will also be creating smaller dataframes for each table to pull data from so we can only use the data from each table we want. 

In [5]:
# patients table
conn.execute("""
CREATE TABLE patients (
patient_id INT PRIMARY KEY,
first_name TEXT,
last_name TEXT,
gender TEXT,
date_of_birth TEXT
);
""")

In [6]:
patients_sql = patients_df[['patient_id', 'first_name', 'last_name', 'gender', 'date_of_birth']]

In [7]:
patients_sql.head()

,patient_id,first_name,last_name,gender,date_of_birth
0,P001,David,Williams,F,1955-06-04
1,P002,Emily,Smith,F,1984-10-12
2,P003,Laura,Jones,M,1977-08-21
3,P004,Michael,Johnson,F,1981-02-20
4,P005,David,Wilson,M,1960-06-23


In [8]:
# doctors table
conn.execute("""
CREATE TABLE doctors (
doctor_id INT PRIMARY KEY,
first_name TEXT,
last_name TEXT,
specialization TEXT
);
""")

In [9]:
doctors_sql = doctors_df [['doctor_id', 'first_name', 'last_name', 'specialization']]

In [10]:
doctors_sql.head()

,doctor_id,first_name,last_name,specialization
0,D001,David,Taylor,Dermatology
1,D002,Jane,Davis,Pediatrics
2,D003,Jane,Smith,Pediatrics
3,D004,David,Jones,Pediatrics
4,D005,Sarah,Taylor,Dermatology


In [11]:
# appointmets table
conn.execute("""
CREATE TABLE appointments (
appointment_id INT PRIMARY KEY,
patient_id INT,
doctor_id INT,
appointment_date TEXT,
status TEXT,

FOREIGN KEY(patient_id)
    REFERENCES patients(patient_id)
FOREIGN KEY(doctor_id)
    REFERENCES doctors(doctor_id)
)
""")

In [12]:
appointments_sql = appointments_df [['appointment_id', 'patient_id', 'doctor_id', 'appointment_date', 'status']]

In [13]:
appointments_sql.head()

,appointment_id,patient_id,doctor_id,appointment_date,status
0,A001,P034,D009,2023-08-09,Scheduled
1,A002,P032,D004,2023-06-09,No-show
2,A003,P048,D004,2023-06-28,Cancelled
3,A004,P025,D006,2023-09-01,Cancelled
4,A005,P040,D003,2023-07-06,No-show


### Data Insertion
Here we are populating our tables, then using the output below the insertion to validate it worked properly.

In [14]:
# patients table
patients_sql.to_sql(
    'patients',
    conn,
    if_exists = 'append',
    index = False
)

50

In [15]:
# validation
pd.read_sql("""
SELECT COUNT(*) as total_patients
FROM patients
""", conn)

,total_patients
0,50


In [16]:
pd.read_sql("""
SELECT *
FROM patients
LIMIT 5
""", conn)

,patient_id,first_name,last_name,gender,date_of_birth
0,P001,David,Williams,F,1955-06-04
1,P002,Emily,Smith,F,1984-10-12
2,P003,Laura,Jones,M,1977-08-21
3,P004,Michael,Johnson,F,1981-02-20
4,P005,David,Wilson,M,1960-06-23


In [17]:
# doctors table
doctors_sql.to_sql(
    'doctors',
    conn,
    if_exists = 'append',
    index = False
)

10

In [18]:
# validation
pd.read_sql("""
SELECT COUNT(*) as total_doctors
FROM doctors
""", conn)

,total_doctors
0,10


In [19]:
pd.read_sql("""
SELECT * 
FROM doctors
LIMIT 5
""", conn)

,doctor_id,first_name,last_name,specialization
0,D001,David,Taylor,Dermatology
1,D002,Jane,Davis,Pediatrics
2,D003,Jane,Smith,Pediatrics
3,D004,David,Jones,Pediatrics
4,D005,Sarah,Taylor,Dermatology


In [20]:
# appointments table
appointments_sql.to_sql(
    'appointments',
    conn,
    if_exists = 'append',
    index = False
)

200

In [21]:
# validation
pd.read_sql("""
SELECT COUNT(*) as total_appointments
FROM appointments
""", conn)

,total_appointments
0,200


In [22]:
pd.read_sql("""
select *
FROM appointments
LIMIT 5
""", conn)

,appointment_id,patient_id,doctor_id,appointment_date,status
0,A001,P034,D009,2023-08-09,Scheduled
1,A002,P032,D004,2023-06-09,No-show
2,A003,P048,D004,2023-06-28,Cancelled
3,A004,P025,D006,2023-09-01,Cancelled
4,A005,P040,D003,2023-07-06,No-show
